# 03 - Análisis exploratorio

Preguntas guía: ¿cómo se distribuye el consumo?, ¿cómo se relacionan edad y consumo?, ¿qué cambia cuando agregamos soporte y plan al análisis?


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
ROOT = Path("..").resolve()

df = pd.read_csv(ROOT / "data" / "processed" / "streaming_users_processed.csv")
df.head()


In [ ]:
df[["age", "monthly_watch_time_mins", "customer_support_tickets"]].describe().round(2)


## Análisis univariado: distribución del tiempo mensual

Permite evaluar el comportamiento típico de consumo y si la winsorización dejó una escala interpretable.


In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(df["monthly_watch_time_mins"], bins=35, kde=True, color="#2f6f73")
plt.title("Distribución del tiempo mensual de visualización")
plt.xlabel("Minutos mensuales")
plt.ylabel("Usuarios")
plt.show()


Interpretación: la mayoría de usuarios se concentra en consumos medios, con una cola derecha moderada. Esto sugiere que existen usuarios intensivos, pero el consumo extremo original ya no domina la lectura.


## Análisis multivariado de 2 variables: edad y consumo

Evalúa si usuarios de distintas edades muestran cambios en el tiempo mensual de visualización.


In [ ]:
plt.figure(figsize=(8,5))
sns.regplot(data=df, x="age", y="monthly_watch_time_mins", scatter_kws={"alpha":0.25, "s":18}, line_kws={"color":"#d95f02"})
plt.title("Relación entre edad y tiempo mensual de visualización")
plt.xlabel("Edad")
plt.ylabel("Minutos mensuales")
plt.show()
print("Correlación edad-consumo:", round(df["age"].corr(df["monthly_watch_time_mins"]), 3))


Interpretación: este gráfico permite evaluar si el consumo aumenta o disminuye con la edad. Si la correlación es baja, la edad por sí sola no explica el nivel de consumo y conviene incorporar otras variables.


## Análisis multivariado de 3 variables: edad, consumo y soporte por plan

Agrega `customer_support_tickets` como tamaño del punto y `subscription_plan` como color para observar perfiles más completos.


In [ ]:
plt.figure(figsize=(9,6))
sns.scatterplot(
    data=df,
    x="age",
    y="monthly_watch_time_mins",
    hue="subscription_plan",
    size="customer_support_tickets",
    sizes=(20, 180),
    alpha=0.45
)
plt.title("Edad, consumo mensual y tickets de soporte por plan")
plt.xlabel("Edad")
plt.ylabel("Minutos mensuales")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()


Interpretación: al sumar plan y tickets se observa si los usuarios con mayor consumo también concentran más consultas de soporte y si ese patrón cambia entre planes. Esto es multivariado porque la lectura depende simultáneamente de edad, consumo, soporte y segmento de plan.


## Análisis multivariado adicional: edad, consumo y género favorito

Este cruce permite ver si los patrones de consumo por edad cambian según el contenido preferido por los usuarios.


In [ ]:
plt.figure(figsize=(10,6))
sns.scatterplot(
    data=df,
    x="age",
    y="monthly_watch_time_mins",
    hue="favorite_genre",
    alpha=0.38,
    s=28
)
plt.title("Edad y consumo mensual según género favorito")
plt.xlabel("Edad")
plt.ylabel("Minutos mensuales")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()


In [ ]:
resumen_genero = (
    df.groupby("favorite_genre")
      .agg(
          usuarios=("user_id", "count"),
          edad_mediana=("age", "median"),
          consumo_mediano=("monthly_watch_time_mins", "median"),
          consumo_promedio=("monthly_watch_time_mins", "mean")
      )
      .sort_values("consumo_mediano", ascending=False)
      .round(2)
)
resumen_genero


Interpretación: si un género muestra mayor consumo mediano, puede indicar una preferencia asociada a usuarios más intensivos. La edad mediana ayuda a distinguir si esa diferencia parece vinculada al perfil etario o al tipo de contenido preferido.


## Apoyo multivariado: matriz de correlación

Resume las relaciones lineales entre las variables numéricas usadas en el EDA y PCA.


In [ ]:
corr = df[["age", "monthly_watch_time_mins", "customer_support_tickets"]].corr()
plt.figure(figsize=(6,4))
sns.heatmap(corr, annot=True, cmap="vlag", vmin=-1, vmax=1)
plt.title("Correlación entre edad, consumo y soporte")
plt.show()


Interpretación: el heatmap ayuda a confirmar si las relaciones observadas visualmente son fuertes o débiles. Correlación no implica causalidad; funciona como evidencia exploratoria.
